In [1]:
import torch
import torch.nn as nn
import torchaudio
import soundfile as sf
import numpy as np
import pandas as pd
import json
import mlflow
import mlflow.pytorch
from pathlib import Path
from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device : {DEVICE}")
print(f"   torch  : {torch.__version__}")

✅ Device : cpu
   torch  : 2.10.0+cpu


In [2]:
BASE_DIR       = Path("../")
PROCESSED_DIR  = BASE_DIR / "data/processed"
CHECKPOINT_DIR = BASE_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(exist_ok=True)

print(f"Processed dir  : {PROCESSED_DIR.exists()}")
print(f"Checkpoint dir : {CHECKPOINT_DIR.exists()}")

Processed dir  : True
Checkpoint dir : True


In [3]:
df = pd.read_csv(PROCESSED_DIR / "metadata.csv")
df = df[df['word'] != ''].reset_index(drop=True)

with open(PROCESSED_DIR / "vocab.json", "r", encoding="utf-8") as f:
    char2idx = json.load(f)

idx2char   = {i: ch for ch, i in char2idx.items()}
VOCAB_SIZE = len(char2idx)

print(f"✅ Samples    : {len(df)}")
print(f"✅ Vocab size : {VOCAB_SIZE}")

✅ Samples    : 2500
✅ Vocab size : 41


In [4]:
def extract_mel(wav_path, sample_rate=16000, n_mels=80, n_fft=400, hop_length=160):
    audio_data, sr = sf.read(wav_path)
    if len(audio_data.shape) > 1:
        audio_data = audio_data.mean(axis=1)
    waveform = torch.tensor(audio_data, dtype=torch.float32).unsqueeze(0)
    if sr != sample_rate:
        waveform = torchaudio.transforms.Resample(sr, sample_rate)(waveform)
    mel = torchaudio.transforms.MelSpectrogram(
        sample_rate=sample_rate, n_fft=n_fft,
        hop_length=hop_length, n_mels=n_mels
    )(waveform)
    mel_db = torchaudio.transforms.AmplitudeToDB()(mel)
    return mel_db.squeeze(0)

print("✅ Feature extraction ready")

✅ Feature extraction ready


In [9]:
class UrduASRDataset(Dataset):
    def __init__(self, dataframe, char2idx):
        self.df       = dataframe.reset_index(drop=True)
        self.char2idx = char2idx

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row     = self.df.iloc[idx]
        feature = extract_mel(row['file_path'])
        word    = str(row['word']) if not pd.isna(row['word']) else ''
        label   = [self.char2idx[ch] for ch in word if ch in self.char2idx]
        # fallback if label is empty
        if len(label) == 0:
            label = [0]
        return feature, torch.tensor(label, dtype=torch.long)


def collate_fn(batch):
    features, labels = zip(*batch)
    feat_lengths  = [f.shape[1] for f in features]
    max_feat_len  = max(feat_lengths)
    padded_feats  = torch.zeros(len(features), features[0].shape[0], max_feat_len)
    for i, f in enumerate(features):
        padded_feats[i, :, :f.shape[1]] = f
    label_lengths = [len(l) for l in labels]
    max_label_len = max(label_lengths)
    padded_labels = torch.zeros(len(labels), max_label_len, dtype=torch.long)
    for i, l in enumerate(labels):
        padded_labels[i, :len(l)] = l
    return (padded_feats,
            padded_labels,
            torch.tensor(feat_lengths),
            t

SyntaxError: incomplete input (74754574.py, line 35)

In [6]:
class UrduASRModel(nn.Module):
    def __init__(self, n_mels=80, vocab_size=VOCAB_SIZE,
                 rnn_hidden=256, rnn_layers=3, dropout=0.3):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(dropout)
        )

        cnn_out_size = (n_mels // 2) * 32

        self.rnn = nn.GRU(
            input_size=cnn_out_size,
            hidden_size=rnn_hidden,
            num_layers=rnn_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )

        self.fc = nn.Linear(rnn_hidden * 2, vocab_size)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.cnn(x)
        b, c, f, t = x.shape
        x = x.permute(0, 3, 1, 2)
        x = x.reshape(b, t, c * f)
        x, _ = self.rnn(x)
        x = self.fc(x)
        x = x.permute(1, 0, 2)
        return x

model = UrduASRModel().to(DEVICE)
dummy = torch.zeros(2, 80, 50).to(DEVICE)
out   = model(dummy)
print(f"✅ Model output shape : {out.shape}  (time, batch, vocab_size)")
print(f"   Total parameters  : {sum(p.numel() for p in model.parameters()):,}")

✅ Model output shape : torch.Size([25, 2, 41])  (time, batch, vocab_size)
   Total parameters  : 4,758,537


In [7]:
def train(model, train_loader, val_loader, epochs=30, lr=1e-3):

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
    ctc_loss  = nn.CTCLoss(blank=0, zero_infinity=True)
    best_val  = float('inf')

    mlflow.set_experiment("urdu_asr")

    with mlflow.start_run():
        mlflow.log_params({
            "epochs"     : epochs,
            "lr"         : lr,
            "batch_size" : train_loader.batch_size,
            "rnn_hidden" : 256,
            "rnn_layers" : 3,
            "vocab_size" : VOCAB_SIZE
        })

        for epoch in range(epochs):
            # Training
            model.train()
            train_loss = 0
            for feats, labels, feat_lens, label_lens in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
                feats  = feats.to(DEVICE)
                labels = labels.to(DEVICE)
                optimizer.zero_grad()
                outputs    = model(feats)
                log_probs  = torch.nn.functional.log_softmax(outputs, dim=2)
                input_lens = (feat_lens // 2).clamp(min=1)
                loss       = ctc_loss(log_probs, labels, input_lens, label_lens)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                optimizer.step()
                train_loss += loss.item()
            train_loss /= len(train_loader)

            # Validation
            model.eval()
            val_loss = 0
            with torch.no_grad():
                for feats, labels, feat_lens, label_lens in val_loader:
                    feats      = feats.to(DEVICE)
                    labels     = labels.to(DEVICE)
                    outputs    = model(feats)
                    log_probs  = torch.nn.functional.log_softmax(outputs, dim=2)
                    input_lens = (feat_lens // 2).clamp(min=1)
                    loss       = ctc_loss(log_probs, labels, input_lens, label_lens)
                    val_loss  += loss.item()
            val_loss /= len(val_loader)
            scheduler.step(val_loss)

            print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
            mlflow.log_metrics({"train_loss": train_loss, "val_loss": val_loss}, step=epoch)

            if val_loss < best_val:
                best_val = val_loss
                torch.save(model.state_dict(), CHECKPOINT_DIR / "best_model.pt")
                print(f"   💾 Best model saved (val_loss={best_val:.4f})")

        mlflow.pytorch.log_model(model, "model")
        print(f"\n✅ Training complete. Best val loss: {best_val:.4f}")

print("✅ Training function ready")

✅ Training function ready


In [8]:
model = UrduASRModel().to(DEVICE)
train(model, train_loader, val_loader, epochs=30, lr=1e-3)

2026/03/05 12:13:26 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/05 12:13:26 INFO mlflow.store.db.utils: Updating database tables
2026/03/05 12:13:28 INFO mlflow.tracking.fluent: Experiment with name 'urdu_asr' does not exist. Creating a new experiment.


Epoch 1/30:   0%|          | 0/125 [00:00<?, ?it/s]

TypeError: 'float' object is not iterable